# Lightweight XGBoost: Heart Disease Classification

Train a compact XGBoost model to predict whether a record indicates heart disease (`num > 0`). This beginner-friendly example uses shallow trees, histogram-based training, and one CPU worker. It is for learning only, not for medical diagnosis.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import xgboost as xgb
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    RocCurveDisplay,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_STATE = 42
print(f"XGBoost version: {xgb.__version__}")

## 1. Load and prepare the data

The source target `num` ranges from 0 to 4. We treat 0 as no disease and any value above 0 as disease. Identifier and source columns are excluded from model features.

In [ ]:
data_path_candidates = [
    Path("classification") / "heart_disease_uci.csv",
    Path("..") / "classification" / "heart_disease_uci.csv",
]
data_path = next((path for path in data_path_candidates if path.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "Could not find classification/heart_disease_uci.csv. "
        "Run this notebook from the repository root or the 'xg boost' folder."
    )

data = pd.read_csv(data_path)
if "num" not in data.columns:
    raise ValueError("Expected the target column 'num' in the heart-disease dataset.")

data = data.dropna(subset=["num"]).copy()
data["target"] = (data["num"] > 0).astype(int)
excluded_columns = {"id", "dataset", "num", "target"}
feature_columns = [column for column in data.columns if column not in excluded_columns]
if not feature_columns:
    raise ValueError("No usable feature columns were found.")

X = data[feature_columns]
y = data["target"]
numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()

print(f"Loaded {len(data)} rows from {data_path}")
print(f"Features: {len(feature_columns)}")
display(y.value_counts().sort_index().rename(index={0: "No disease", 1: "Disease"}).to_frame("count"))

## 2. Split the data

A stratified split keeps the class proportions similar in training and testing sets. Preprocessing is fitted only on training data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)
print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")

## 3. Preprocess and train the lightweight model

Missing numeric values are filled with the training median, and missing categories with the most frequent value. Categories are one-hot encoded. The model's small tree count and depth, histogram algorithm, and single worker reduce training time and resource use.

In [ ]:
numeric_preprocessing = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])
categorical_preprocessing = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_preprocessing, numeric_features),
        ("categorical", categorical_preprocessing, categorical_features),
    ],
)

model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    n_estimators=100,
    max_depth=2,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    max_bin=64,
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=1,
)

classifier = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model),
])
classifier.fit(X_train, y_train)
print("Lightweight XGBoost training complete.")

## 4. Evaluate on the held-out test set

Review several metrics rather than accuracy alone. In this educational health-related example, precision and recall describe different types of classification errors.

In [ ]:
y_pred = classifier.predict(X_test)
y_probability = classifier.predict_proba(X_test)[:, 1]

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1 score": f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_probability),
}
display(pd.DataFrame.from_dict(metrics, orient="index", columns=["test score"]).round(3))

print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1],
    target_names=["No disease", "Disease"],
    zero_division=0,
))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=["No disease", "Disease"], cmap="Blues"
)
plt.title("Lightweight XGBoost: test-set confusion matrix")
plt.tight_layout()
plt.show()

RocCurveDisplay.from_predictions(y_test, y_probability)
plt.title("Lightweight XGBoost: test-set ROC curve")
plt.tight_layout()
plt.show()

## 5. Inspect feature importance and predict a sample

Feature importance shows how the fitted trees used features; it does not show cause and effect. The sample prediction below is taken from the test set.

In [ ]:
preprocessor_fitted = classifier.named_steps["preprocessor"]
model_fitted = classifier.named_steps["model"]
feature_names = preprocessor_fitted.get_feature_names_out()
feature_importance = (
    pd.Series(model_fitted.feature_importances_, index=feature_names)
    .sort_values(ascending=False)
    .head(15)
    .sort_values()
)
feature_importance.plot(kind="barh", figsize=(9, 6), title="Top feature importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

sample_patient = X_test.iloc[[0]]
predicted_class = int(classifier.predict(sample_patient)[0])
disease_probability = float(classifier.predict_proba(sample_patient)[0, 1])
print(f"Predicted class: {predicted_class} (0 = no disease, 1 = disease)")
print(f"Estimated disease probability: {disease_probability:.1%}")
print(f"Actual class: {int(y_test.iloc[0])}")

## Recap

This notebook loads the repository's heart-disease data, splits it without leaking test data into preprocessing, trains a compact XGBoost classifier, and evaluates its predictions. This example is for education only and must not be used to diagnose or make health decisions.